# Grafos y modelos generativos

Redes de grafos (GCN), fusión condicional y generación multimodal (HCMG), inversión de gradiente.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

> **Medido en el banco** (`laboratorio/banco/`): ver la sección «cuaderno 07» de `laboratorio/banco/README.md`. La parte generativa (HCMG, D-ReG) se mide al final, junto con los demás generativos.

Medido en el banco (`laboratorio/banco/generativos_07.py`, `resultados-07-generativos.md`): **le gana a una U-Net** al dibujar un dígito nítido según su clase y color (20,44 dB contra 20,04). Es la pista más fuerte del laboratorio.

Medido en el banco: no le gana a una GRU del mismo tamaño (perplejidad 176 contra 146). El modelo solo generaba; se le agregó el entrenamiento.

## Normalización adaptativa y fusión condicional (HCMG)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AdaptiveNormalization(nn.Module):
    """
    Normalización Adaptativa basada en un vector de condición.
    Similar a SPADE o AdaIN, pero simplificado.
    """
    def __init__(self, num_features, cond_dim):
        super().__init__()
        self.norm = nn.InstanceNorm2d(num_features, affine=False) # Normalización sin parámetros aprendibles
        # Capas lineales para predecir los parámetros de escala (gamma) y bias (beta)
        self.gamma_proj = nn.Linear(cond_dim, num_features)
        self.beta_proj = nn.Linear(cond_dim, num_features)

    def forward(self, x, cond_vector):
        # x: activations [batch, channels, H, W]
        # cond_vector: conditioning vector [batch, cond_dim]

        out = self.norm(x) # Normalizar sin affine

        # Predecir gamma y beta a partir del vector de condición
        gamma = self.gamma_proj(cond_vector).unsqueeze(-1).unsqueeze(-1) # [batch, channels, 1, 1]
        beta = self.beta_proj(cond_vector).unsqueeze(-1).unsqueeze(-1)   # [batch, channels, 1, 1]

        # Aplicar escala y bias adaptativos
        out = out * (1 + gamma) + beta
        return out

class ConditionalFusionBlock(nn.Module):
    """
    Bloque de Fusión Condicional (BFC).
    Combina activaciones con vectores condicionales (imagen + texto).
    """
    def __init__(self, in_channels, out_channels, img_id_cond_dim, text_cond_dim):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()

        # Normalización adaptativa para la condición de imagen/ID
        self.norm_img_id = AdaptiveNormalization(out_channels, img_id_cond_dim)
        # Normalización adaptativa para la condición de texto
        self.norm_text = AdaptiveNormalization(out_channels, text_cond_dim)

    def forward(self, x, img_id_vector, text_cond_vector):
        # x: mapa de características de la capa anterior
        # img_id_vector: vector de características de imagen + ID
        # text_cond_vector: vector de características de texto

        out = self.conv(x)

        # Aplicar normalización adaptativa secuencialmente
        # Primero la condición de imagen/ID, luego la de texto
        out = self.norm_img_id(out, img_id_vector)
        out = self.norm_text(out, text_cond_vector)

        out = self.relu(out)
        return out

class HCMG(nn.Module):
    """
    Generador Condicionado Multimodal y Jerárquico (HCMG).
    """
    def __init__(self, input_image_channels, output_image_channels,
                 img_id_cond_dim, text_cond_dim, base_channels=64):
        super().__init__()

        # Capa de entrada de píxeles
        self.initial_conv = nn.Conv2d(input_image_channels, base_channels, kernel_size=3, padding=1)

        # Bloques de Fusión Condicional
        # Podríamos tener BFCs en diferentes resoluciones o profundidades
        self.bfc1 = ConditionalFusionBlock(base_channels, base_channels * 2, img_id_cond_dim, text_cond_dim)
        self.downsample1 = nn.Conv2d(base_channels * 2, base_channels * 2, kernel_size=4, stride=2, padding=1) # Downsample para aumentar features

        self.bfc2 = ConditionalFusionBlock(base_channels * 2, base_channels * 4, img_id_cond_dim, text_cond_dim)
        self.downsample2 = nn.Conv2d(base_channels * 4, base_channels * 4, kernel_size=4, stride=2, padding=1)

        self.bfc3 = ConditionalFusionBlock(base_channels * 4, base_channels * 8, img_id_cond_dim, text_cond_dim)

        # Decodificador (para generar la imagen)
        self.upsample1 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.gen_conv1 = ConditionalFusionBlock(base_channels * 4, base_channels * 2, img_id_cond_dim, text_cond_dim) # Re-inyectar condiciones aquí

        self.upsample2 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.gen_conv2 = ConditionalFusionBlock(base_channels, base_channels, img_id_cond_dim, text_cond_dim) # Re-inyectar condiciones aquí

        self.final_conv = nn.Conv2d(base_channels, output_image_channels, kernel_size=3, padding=1)
        self.tanh = nn.Tanh() # Para salida de imagen (-1 a 1)

    def forward(self, input_pixels, img_id_vector, text_cond_vector):
        # input_pixels: [batch, input_image_channels, H, W]
        # img_id_vector: [batch, img_id_cond_dim]
        # text_cond_vector: [batch, text_cond_dim]

        # Codificación inicial de píxeles
        x = self.initial_conv(input_pixels) # [batch, base_channels, H, W]

        # Paso por BFCs y downsampling
        x = self.bfc1(x, img_id_vector, text_cond_vector) # [batch, base_channels*2, H, W]
        x = self.downsample1(x) # [batch, base_channels*2, H/2, W/2]

        x = self.bfc2(x, img_id_vector, text_cond_vector) # [batch, base_channels*4, H/2, W/2]
        x = self.downsample2(x) # [batch, base_channels*4, H/4, W/4]

        x = self.bfc3(x, img_id_vector, text_cond_vector) # [batch, base_channels*8, H/4, W/4]

        # Decodificación y upsampling con re-inyección condicional
        x = self.upsample1(x)
        x = self.gen_conv1(x, img_id_vector, text_cond_vector) # [batch, base_channels*2, H/2, W/2]

        x = self.upsample2(x)
        x = self.gen_conv2(x, img_id_vector, text_cond_vector) # [batch, base_channels, H, W]

        # Capa de salida
        output_image = self.tanh(self.final_conv(x))
        return output_image

# --- Ejemplo de Uso ---
input_channels = 3 # Por ejemplo, una imagen de contorno o una imagen muy borrosa
output_channels = 3 # Salida de imagen RGB
img_id_cond_dim = 128 # Dimensión del vector de características de imagen + ID (ej. ResNet features + one-hot ID)
text_cond_dim = 768 # Dimensión de embedding de texto (ej. de un Transformer de texto como BERT o CLIP)

model = HCMG(input_channels, output_channels, img_id_cond_dim, text_cond_dim)
print(model)

# Crear inputs dummy
dummy_input_pixels = torch.randn(4, 3, 64, 64) # Batch de 4 imágenes 64x64
dummy_img_id_vector = torch.randn(4, img_id_cond_dim) # 4 vectores de condición de imagen
dummy_text_cond_vector = torch.randn(4, text_cond_dim) # 4 vectores de condición de texto

# Pasar por el modelo
output_image = model(dummy_input_pixels, dummy_img_id_vector, dummy_text_cond_vector)
print(f"\nForma de la imagen de salida: {output_image.shape}")

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Número total de parámetros: {total_params}")

## Redes de grafos de la industria (GCN y GAT con Cora)

Referencia: los modelos de grafos que se usan hoy.

In [ ]:
# --- Configuración del entorno y PyTorch Geometric ---
# Descomentar y ejecutar en Colab si no tienes las librerías
import os
!pip install -q torch
os.environ['TORCH'] = torch.__version__
os.environ['PYTHONWARNINGS'] = "ignore"
!pip install torch_geometric

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt
import torch
import os
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv, GATConv
import numpy as np


# Código para el grafo de NetworkX (Introducción)
H = nx.DiGraph()

# Adding nodes
H.add_nodes_from([
  (0, {"color": "blue", "size": 250}),
  (1, {"color": "yellow", "size": 400}),
  (2, {"color": "orange", "size": 150}),
  (3, {"color": "red", "size": 600})
])

# Adding edges
H.add_edges_from([
  (0, 1),
  (1, 2),
  (1, 0),
  (1, 3),
  (2, 3),
  (3,0)
])

node_colors = nx.get_node_attributes(H, "color").values()
colors = list(node_colors)
node_sizes = nx.get_node_attributes(H, "size").values()
sizes = list(node_sizes)

# Plotting Graph (Directed)
nx.draw(H, with_labels=True, node_color=colors, node_size=sizes)
plt.title("Grafo Dirigido (H)")
plt.show() # Asegúrate de mostrar el primer grafo

G = H.to_undirected()
# Plotting Graph (Undirected)
nx.draw(G, with_labels=True, node_color=colors, node_size=sizes)
plt.title("Grafo No Dirigido (G)")
plt.show() # Asegúrate de mostrar el segundo grafo

# --- Carga del Dataset Planetoid Cora ---
dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())

print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.
print(data)


# --- Implementación del Modelo GCN (Graph Convolutional Network) ---
class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GCNConv(dataset.num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, dataset.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model_gcn = GCN(hidden_channels=16)
print("Modelo GCN:")
print(model_gcn)


# --- Función para Visualización con t-SNE ---
# %matplotlib inline
def visualize(h, color, title="Visualización de Nodos"):
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])
    plt.title(title)
    plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    plt.show()

# --- Entrenamiento del Modelo GCN ---
optimizer_gcn = torch.optim.Adam(model_gcn.parameters(), lr=0.01, weight_decay=5e-4)
criterion_gcn = torch.nn.CrossEntropyLoss()

def train_gcn():
      model_gcn.train()
      optimizer_gcn.zero_grad()
      out = model_gcn(data.x, data.edge_index)
      loss = criterion_gcn(out[data.train_mask], data.y[data.train_mask])
      loss.backward()
      optimizer_gcn.step()
      return loss

def test_gcn():
      model_gcn.eval()
      out = model_gcn(data.x, data.edge_index)
      pred = out.argmax(dim=1)
      test_correct = pred[data.test_mask] == data.y[data.test_mask]
      test_acc = int(test_correct.sum()) / int(data.test_mask.sum())
      return test_acc

print("\n--- Entrenamiento del Modelo GCN ---")
for epoch in range(1, 101):
    loss = train_gcn()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

test_acc_gcn = test_gcn()
print(f'Test Accuracy (GCN): {test_acc_gcn:.4f}')

model_gcn.eval()
out_gcn = model_gcn(data.x, data.edge_index)
visualize(out_gcn, color=data.y, title="Visualización de Nodos después de GCN")


# --- Implementación del Modelo GAT (Graph Attention Network) ---
class GAT(torch.nn.Module):
    def __init__(self, hidden_channels, heads):
        super().__init__()
        torch.manual_seed(1234567)
        self.conv1 = GATConv(dataset.num_features, hidden_channels, heads)
        # La segunda capa toma 'heads * hidden_channels' como entrada
        self.conv2 = GATConv(heads * hidden_channels, dataset.num_classes, heads=1) # Generalmente la última capa de GAT usa heads=1 para la clasificación

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x) # Función de activación ELU es común en GAT
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# Nota: El código original tenía un print de GAT() antes de la definición completa, lo he movido.
model_gat = GAT(hidden_channels=8, heads=8)
print("\nModelo GAT:")
print(model_gat)

# --- Entrenamiento del Modelo GAT ---
optimizer_gat = torch.optim.Adam(model_gat.parameters(), lr=0.005, weight_decay=5e-4)
criterion_gat = torch.nn.CrossEntropyLoss()

def train_gat():
      model_gat.train()
      optimizer_gat.zero_grad()
      out = model_gat(data.x, data.edge_index)
      loss = criterion_gat(out[data.train_mask], data.y[data.train_mask])
      loss.backward()
      optimizer_gat.step()
      return loss

def test_gat(mask):
      model_gat.eval()
      out = model_gat(data.x, data.edge_index)
      pred = out.argmax(dim=1)
      correct = pred[mask] == data.y[mask]
      acc = int(correct.sum()) / int(mask.sum())
      return acc

val_acc_all_gat = []
test_acc_all_gat = []

print("\n--- Entrenamiento del Modelo GAT ---")
for epoch in range(1, 101):
    loss = train_gat()
    val_acc = test_gat(data.val_mask)
    test_acc = test_gat(data.test_mask)
    val_acc_all_gat.append(val_acc)
    test_acc_all_gat.append(test_acc)
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}')


# --- Visualización de resultados de GAT ---
plt.figure(figsize=(12,8))
plt.plot(np.arange(1, len(val_acc_all_gat) + 1), val_acc_all_gat, label='Validation accuracy', c='blue')
plt.plot(np.arange(1, len(test_acc_all_gat) + 1), test_acc_all_gat, label='Testing accuracy', c='red')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('GATConv Accuracy over Epochs')
plt.legend(loc='lower right', fontsize='x-large')
# plt.savefig('gat_loss.png') # Descomentar para guardar la imagen
plt.show()

model_gat.eval()
out_gat = model_gat(data.x, data.edge_index)
visualize(out_gat, color=data.y, title="Visualización de Nodos después de GAT")

In [ ]:
import torch
from torch_geometric.data import Data

# --- Definir las características de tus "neuronas" ---

# Ejemplo:
# Neurona 0: Capa de Entrada, posición (0.0, 1.0)
# Neurona 1: Capa de Entrada, posición (0.0, 2.0)
# Neurona 2: Capa Oculta, posición (1.0, 1.5)
# Neurona 3: Capa de Salida, posición (2.0, 1.5)

# Características para data.x:
# Columna 0: Tipo de neurona (ej. 0=entrada, 1=oculta, 2=salida) - One-hot encoding podría ser mejor
# Columna 1: Posición X
# Columna 2: Posición Y
# Columna 3: Otra característica (ej. Valor de activación inicial si aplica, o un ID embebido)
# ... puedes agregar más características relevantes para tu problema

# Usaremos un encoding simple para el tipo (0, 1, 2). Para un uso real, one-hot encoding sería mejor.
# Por ejemplo, si el tipo de neurona es una característica categórica, puedes usar one-hot encoding:
# Tipo de neurona: [es_entrada, es_oculta, es_salida]
# 0: entrada -> [1, 0, 0]
# 1: oculta  -> [0, 1, 0]
# 2: salida  -> [0, 0, 1]

# Ejemplo con one-hot encoding para el tipo de neurona:
# Dimensión de características: 3 (para tipo) + 2 (para posición) + 1 (para otra_info) = 6
features_neuronas = torch.tensor([
    # Tipo       Pos X  Pos Y  Otra info
    [1.0, 0.0, 0.0, 0.0,  1.0,   0.5], # Neurona 0 (Entrada)
    [1.0, 0.0, 0.0, 0.0,  2.0,   0.8], # Neurona 1 (Entrada)
    [0.0, 1.0, 0.0, 1.0,  1.5,   0.2], # Neurona 2 (Oculta)
    [0.0, 0.0, 1.0, 2.0,  1.5,   0.1]  # Neurona 3 (Salida)
], dtype=torch.float)

# --- Definir las conexiones (aristas) entre tus "neuronas" ---
# Las conexiones aquí serían las conexiones de la red neuronal tradicional.
# Ejemplo:
# De Neurona 0 (entrada) a Neurona 2 (oculta)
# De Neurona 1 (entrada) a Neurona 2 (oculta)
# De Neurona 2 (oculta) a Neurona 3 (salida)

edge_index_neuronas = torch.tensor([
    [0, 1, 2], # Origen de la conexión (índices de neuronas)
    [2, 2, 3]  # Destino de la conexión (índices de neuronas)
], dtype=torch.long)

# --- Crear el objeto Data de PyTorch Geometric ---
# Si tu objetivo es clasificar estas "neuronas" (ej. en tipos), podrías añadir data.y
# Por ejemplo, si la tarea es clasificar la "etapa" de la neurona (entrada, oculta, salida)
labels_neuronas = torch.tensor([0, 0, 1, 2], dtype=torch.long) # 0=entrada, 1=oculta, 2=salida

# Crear el objeto Data
data_red_neuronal_como_grafo = Data(
    x=features_neuronas,
    edge_index=edge_index_neuronas,
    y=labels_neuronas # Opcional, si tienes etiquetas
)

print(data_red_neuronal_como_grafo)
print(f"Características de las neuronas (data.x):\n{data_red_neuronal_como_grafo.x}")
print(f"Conexiones de las neuronas (data.edge_index):\n{data_red_neuronal_como_grafo.edge_index}")
print(f"Etiquetas de las neuronas (data.y):\n{data_red_neuronal_como_grafo.y}")

## Neuronas espaciales

Neuronas con posición en el espacio que se conectan y se pasan señales (simulación, sin entrenamiento), y su visualización con embeddings de BERT.

In [ ]:
# prompt: ## Seudocódigo para Neuronas Espaciales (sin capas explícitas)
# # 1. Definición de la Neurona
# Clase Neurona:
#     Atributos:
#         id: Identificador único de la neurona
#         posicion: Una tupla o lista que representa las coordenadas (x, y, z, ...)
#         estado_activacion: Valor numérico (ej. 0 o 1, o un valor continuo)
#         conexiones_salientes: Lista de tuplas (neurona_destino, peso)
#         umbral_activacion: Valor que debe superar la suma ponderada para activarse
#     Métodos:
#         __init__(id, posicion, umbral):
#             Inicializar id, posicion, umbral_activacion
#             estado_activacion = 0
#             conexiones_salientes = []
#         añadir_conexion(neurona_destino, peso):
#             Añadir (neurona_destino, peso) a conexiones_salientes
#         calcular_activacion(entradas_recibidas):
#             # entradas_recibidas es un diccionario {neurona_origen_id: valor_señal}
#             suma_ponderada = 0
#             Para cada neurona_origen_id, valor_señal en entradas_recibidas:
#                 # Aquí se podría considerar la distancia espacial entre neuronas
#                 # para ajustar el peso o la influencia de la señal.
#                 # Por simplicidad, asumimos que entradas_recibidas ya maneja esto
#                 # o es una consideración a aplicar en la propagación.
#                 suma_ponderada = suma_ponderada + valor_señal
#             Si suma_ponderada > umbral_activacion:
#                 estado_activacion = funcion_activacion(suma_ponderada)
#             Sino:
#                 estado_activacion = 0 # O un valor de reposo
#             Retornar estado_activacion
#         propagar_señal():
#             # Distribuye el estado de activación a las neuronas conectadas
#             señales_a_enviar = []
#             Para cada conexion en conexiones_salientes:
#                 neurona_destino, peso = conexion
#                 señal = estado_activacion * peso
#                 Añadir (neurona_destino.id, señal)

# Implementación de la clase Neurona Espacial
class NeuronaEspacial:
    def __init__(self, id_neurona, posicion, dimensiones, umbral_activacion=0.5, tipo="oculta"):
        self.id = id_neurona
        self.posicion = posicion  # Tupla o lista (ej. (x, y, z))
        self.dimensiones = dimensiones # Tupla o lista (ej. (alto, ancho, profundidad))
        self.activacion = 0.0  # Valor numérico del estado de activación
        self.umbral_activacion = umbral_activacion # Umbral para activarse
        self.tipo = tipo # "entrada", "oculta", "salida"

        self.conexiones_salientes = [] # Lista de tuplas (neurona_destino, peso)
        self.conexiones_entrantes = [] # Opcional: lista de (neurona_origen, peso)

    def añadir_conexion(self, neurona_destino, peso=1.0):
        """Añade una conexión saliente a otra neurona."""
        self.conexiones_salientes.append((neurona_destino, peso))

    def recibir_senal(self, valor_senal):
        """Recibe una señal entrante y la suma a la activación."""
        self.activacion += valor_senal

    def calcular_activacion(self, entradas_recibidas=None):
        """
        Calcula la activación basada en las señales recibidas.
        En un modelo simple, puede ser solo una función de activación.
        En modelos más complejos, puede involucrar umbrales o dinámicas.
        """
        # Para este pseudocódigo, asumimos que 'recibir_senal' ya sumó las entradas.
        # Si queremos un umbral, lo aplicamos aquí.
        if self.activacion > self.umbral_activacion:
             # Ejemplo simple de función de activación (relu)
             self.activacion = max(0, self.activacion)
             # O una sigmoide si quieres salida entre 0 y 1
             # self.activacion = 1 / (1 + math.exp(-self.activacion))
        else:
            self.activacion = 0.0 # Si no supera el umbral, se desactiva (o vuelve a un estado base)

        return self.activacion


    def propagar_señal(self):
        """
        Propaga la señal de activación a las neuronas conectadas.
        Retorna una lista de (neurona_destino, señal_enviada).
        """
        senales_a_enviar = []
        if self.activacion > 0: # Solo propaga si está activada
            for neurona_destino, peso in self.conexiones_salientes:
                señal = self.activacion * peso
                senales_a_enviar.append((neurona_destino, señal))
        return senales_a_enviar

# --- Implementación de un Espacio con Neuronas Espaciales ---
class EspacioNeuronal:
    def __init__(self):
        self.neuronas = {} # Diccionario {id_neurona: objeto_neurona}
        self.ultima_id = 0

    def añadir_neurona(self, posicion, dimensiones, umbral_activacion=0.5, tipo="oculta"):
        """Añade una nueva neurona al espacio."""
        self.ultima_id += 1
        new_neurona = NeuronaEspacial(self.ultima_id, posicion, dimensiones, umbral_activacion, tipo)
        self.neuronas[self.ultima_id] = new_neurona
        return new_neurona

    def conectar_neuronas(self, id_origen, id_destino, peso=1.0):
        """Establece una conexión de origen a destino."""
        if id_origen in self.neuronas and id_destino in self.neuronas:
            neurona_origen = self.neuronas[id_origen]
            neurona_destino = self.neuronas[id_destino]
            neurona_origen.añadir_conexion(neurona_destino, peso)
            # Opcional: añadir conexión entrante si el modelo es bidireccional o lo necesita
            # neurona_destino.conexiones_entrantes.append((neurona_origen, peso))
        else:
            print(f"Error: Una o ambas neuronas ({id_origen}, {id_destino}) no existen.")

    def simular_paso_tiempo(self, senales_externas={}):
        """
        Simula un paso de tiempo en el espacio neuronal.
        Procesa señales externas, calcula activaciones y propaga señales.
        senales_externas: {id_neurona_destino: valor_senal}
        """
        # Paso 1: Aplicar señales externas y resetear activación (o decaerla)
        # Resetear la activación para un modelo no recurrente simple
        for neurona in self.neuronas.values():
             neurona.activacion = 0.0

        # Aplicar señales externas
        for id_destino, senal in senales_externas.items():
            if id_destino in self.neuronas:
                self.neuronas[id_destino].recibir_senal(senal)
            else:
                print(f"Advertencia: Señal externa para neurona {id_destino} que no existe.")


        # Paso 2: Recolectar todas las señales a propagar desde neuronas activas del paso anterior (o entradas directas)
        # En este modelo simple, las "señales recibidas" son directas del paso anterior o externas.
        # Las entradas a 'calcular_activacion' en el pseudocódigo original implicaban recolectar señales
        # EN este paso ANTES de recalcular la activación. Adaptamos el flujo:
        # Recolectamos señales de propagación *basadas en la activación ya actualizada por señales externas*
        # Si hubiera activación previa que decae, la consideraríamos aquí.

        senales_propagadas = []
        for neurona in self.neuronas.values():
            # Calcula la activación final después de recibir entradas (externas o de propagación previa)
            # En este simple ejemplo, la 'recibir_senal' ya sumó. Si tuvieras dinámica, la aplicarías aquí.
            # Por simplicidad, asumimos que la activación ya refleja las entradas sumadas.
            # Llama a calcular_activacion para aplicar umbral, etc.
            neurona.calcular_activacion() # Aplica umbral/func.activación a la activación actual

            # Ahora, si la neurona está activada, prepara sus señales de salida
            if neurona.activacion > 0:
                 senales_propagadas.extend(neurona.propagar_señal())


        # Paso 3: Distribuir las señales propagadas a las neuronas destino en el *próximo* paso de tiempo
        # Para simular en un solo paso, podemos aplicarlas inmediatamente, pero conceptualmente son "entradas para el próximo ciclo".
        # Como nuestro modelo simple reinicia activación cada paso, necesitamos acumular las entradas para la próxima ronda
        # o modificar la activación de destino DENTRO de este paso (si la propagación es instantánea)
        # Optamos por la propagación instantánea dentro del mismo paso para simplificar:

        # Reiniciar activaciones para el próximo ciclo antes de aplicar señales propagadas
        # (Este reset dependerá de la dinámica de tu modelo - decaimiento, etc.)
        # En este ejemplo, la propagación es instantánea dentro del mismo paso, así que no reseteamos AHORA.
        # Las señales propagadas de este paso se aplican *inmediatamente* a la activación de los destinos.

        # Ahora, distribuir las señales propagadas
        senales_recibidas_en_este_paso = {} # Acumula señales por destino
        for neurona_destino, senal in senales_propagadas:
             if neurona_destino.id not in senales_recibidas_en_este_paso:
                 senales_recibidas_en_este_paso[neurona_destino.id] = 0
             senales_recibidas_en_este_paso[neurona_destino.id] += senal

        # Aplicar las señales acumuladas a las neuronas destino
        for id_destino, total_senal_recibida in senales_recibidas_en_este_paso.items():
             self.neuronas[id_destino].recibir_senal(total_senal_recibida)


        # Después de aplicar TODAS las señales, las activaciones están actualizadas.
        # El estado de las neuronas al final de esta función es el resultado del paso de simulación.

    def get_estado(self):
        """Retorna un resumen del estado actual de las neuronas."""
        estado = {}
        for id_neurona, neurona in self.neuronas.items():
            estado[id_neurona] = {
                "posicion": neurona.posicion,
                "dimensiones": neurona.dimensiones,
                "activacion": neurona.activacion,
                "tipo": neurona.tipo
            }
        return estado

# --- Ejemplo de uso del EspacioNeuronal ---

# Inicializar un espacio
espacio = EspacioNeuronal()

# Añadir algunas neuronas espaciales
n1 = espacio.añadir_neurona(posicion=(0, 0, 0), dimensiones=(10, 10, 10), tipo="entrada", umbral_activacion=0.1)
n2 = espacio.añadir_neurona(posicion=(5, 5, 5), dimensiones=(8, 8, 8), tipo="oculta", umbral_activacion=0.3)
n3 = espacio.añadir_neurona(posicion=(10, 10, 10), dimensiones=(5, 5, 5), tipo="salida", umbral_activacion=0.5)
n4 = espacio.añadir_neurona(posicion=(5, 0, 10), dimensiones=(6, 6, 6), tipo="oculta", umbral_activacion=0.2)


# Conectar las neuronas
espacio.conectar_neuronas(n1.id, n2.id, peso=0.8)
espacio.conectar_neuronas(n1.id, n4.id, peso=0.6)
espacio.conectar_neuronas(n2.id, n3.id, peso=0.7)
espacio.conectar_neuronas(n4.id, n3.id, peso=0.9)


# Simular pasos de tiempo
print("--- Simulación del Espacio Neuronal ---")

# Paso 1: Aplicar señal externa a la neurona de entrada 1
print("\nPaso 1: Señal externa a Neurona 1")
espacio.simular_paso_tiempo(senales_externas={n1.id: 1.5})
estado_paso1 = espacio.get_estado()
for id, data in estado_paso1.items():
    print(f"Neurona {id}: Activación = {data['activacion']:.4f}")

# Paso 2: Simular otro paso (las activaciones se basan en el estado final del paso 1)
print("\nPaso 2: Simulación (propagación interna)")
# No hay señales externas en este paso, solo propagación interna
espacio.simular_paso_tiempo(senales_externas={})
estado_paso2 = espacio.get_estado()
for id, data in estado_paso2.items():
    print(f"Neurona {id}: Activación = {data['activacion']:.4f}")

# Paso 3: Simular otro paso (propagación)
print("\nPaso 3: Simulación (propagación)")
espacio.simular_paso_tiempo(senales_externas={})
estado_paso3 = espacio.get_estado()
for id, data in estado_paso3.items():
    print(f"Neurona {id}: Activación = {data['activacion']:.4f}")

print("\n--- Fin de la simulación ---")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
from sklearn.decomposition import PCA # Assuming PCA import is intended based on ipython-input-14

# Supongamos que tienes tus neuronas en una lista
# Se asume que la clase NeuronaEspacial está definida en celdas anteriores y es accesible.
# Si no, debes incluir su definición aquí.
# from . import NeuronaEspacial # Ejemplo si estuviera en otro archivo, no aplica en notebook

# Re-definiendo NeuronaEspacial just in case it's not available from previous cells
class NeuronaEspacial:
    def __init__(self, id_neurona, posicion, dimensiones, umbral_activacion=0.5, tipo="oculta"):
        self.id = id_neurona
        self.posicion = posicion  # Tupla o lista (ej. (x, y, z))
        self.dimensiones = dimensiones # Tupla o lista (ej. (alto, ancho, profundidad))
        self.activacion = 0.0  # Valor numérico del estado de activación
        self.umbral_activacion = umbral_activacion # Umbral para activarse
        self.tipo = tipo # "entrada", "oculta", "salida"
        self.conexiones_salientes = []
        self.conexiones_entrantes = []


neuronas = [
    NeuronaEspacial(1, (0, 0, 0), (10, 10, 10), tipo="entrada"),
    NeuronaEspacial(2, (10, 5, 2), (5, 5, 5), tipo="oculta"),
    NeuronaEspacial(3, (2, 8, 12), (8, 8, 8), tipo="salida")
]

# Opcional: darles activaciones distintas para simular estados
neuronas[0].activacion = 1.0
neuronas[1].activacion = 0.7
neuronas[2].activacion = 0.35

# Codificar tipo como valor numérico
tipo_map = {"entrada": 0, "oculta": 1, "salida": 2}

# Vectorizar cada neurona
features = []
for n in neuronas:
    x, y, z = n.posicion
    alto, ancho, prof = n.dimensiones
    tipo_val = tipo_map.get(n.tipo, 1)
    features.append([x, y, z, alto, ancho, prof, n.activacion, tipo_val])

# Convertir a array para t-SNE
X = np.array(features)

# Aplicar t-SNE para reducir a 2D
# CORRECCIÓN: perplexity debe ser menor que el número de muestras (3).
# Cambiamos perplexity de 3 a 1 (o 2 también sería válido).
tsne = TSNE(n_components=2, perplexity=1, learning_rate=200, n_iter=1000, random_state=42)
X_embedded = tsne.fit_transform(X)

# Visualizar
plt.figure(figsize=(8, 6))
for i, coord in enumerate(X_embedded):
    tipo = neuronas[i].tipo
    plt.scatter(coord[0], coord[1], label=f"Neurona {neuronas[i].id} ({tipo})")
    plt.text(coord[0]+0.5, coord[1], f"ID {neuronas[i].id}", fontsize=9)

plt.title("Visualización 2D de Neuronas Espaciales con t-SNE")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.grid(True)
plt.show()

# PCA visualization (kept from original cell)
# Ensure X is defined from the features list above
pca = PCA(n_components=2)
X_pca_embedded = pca.fit_transform(X) # Using a different variable name for PCA results

plt.figure(figsize=(8, 6))
for i, coord in enumerate(X_pca_embedded):
    tipo = neuronas[i].tipo
    plt.scatter(coord[0], coord[1], label=f"Neurona {neuronas[i].id} ({tipo})")
    plt.text(coord[0]+0.5, coord[1], f"ID {neuronas[i].id}", fontsize=9)

plt.title("Visualización 2D de Neuronas Espaciales con PCA")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import random

# 1. Definir Neurona (posición)
class Neurona:
    def __init__(self, id, x, y):
        self.id = id
        self.x = x
        self.y = y
        self.conexiones = [] # Lista de IDs de neuronas conectadas

# 2. Crear Neuronas
num_neuronas = 20
neuronas = []
for i in range(num_neuronas):
    x = random.uniform(0, 10) # Posición aleatoria en el rango [0, 10]
    y = random.uniform(0, 10)
    neuronas.append(Neurona(i, x, y))

# 3. Definir Conexiones (ejemplo: conexiones aleatorias simples)
for i in range(num_neuronas):
    for j in range(i + 1, num_neuronas): # Evitar duplicados y auto-conexiones
        if random.random() < 0.2: # 20% de probabilidad de conexión
            neuronas[i].conexiones.append(neuronas[j].id)
            neuronas[j].conexiones.append(neuronas[i].id) # Conexión bidireccional

# 4. Visualizar
fig, ax = plt.subplots()

# Dibujar neuronas
for n in neuronas:
    ax.plot(n.x, n.y, 'o', color='skyblue', markersize=10) # Círculos para neuronas

# Dibujar conexiones
for n1 in neuronas:
    for conectado_id in n1.conexiones:
        n2 = next((n for n in neuronas if n.id == conectado_id), None)
        if n2:
            ax.plot([n1.x, n2.x], [n1.y, n2.y], '-', color='blue', linewidth=0.5) # Líneas para conexiones
            # Opcional: añadir puntos luminosos en las conexiones
            # ax.plot((n1.x + n2.x) / 2, (n1.y + n2.y) / 2, '*', color='yellow', markersize=5)


ax.set_title("Simulación de Conexiones Neuronales Espaciales")
ax.set_xlabel("Eje X")
ax.set_ylabel("Eje Y")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal', adjustable='box')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

# --- Paso 1: Simular 100 neuronas (cada una con 10 pesos y 1 bias)
n_neuronas = 100
n_entradas = 10

# Pesos y bias aleatorios
pesos = np.random.randn(n_neuronas, n_entradas)
bias = np.random.randn(n_neuronas, 1)

# --- Paso 2: Entrada común para todas las neuronas
entrada = np.random.randn(n_entradas, 1)

# --- Paso 3: Calcular salidas (suma ponderada + ReLU)
activaciones = np.maximum(0, pesos @ entrada + bias)  # ReLU

# --- Paso 4: Usar los pesos+bias como representación de la neurona
vectores_neuronas = np.hstack([pesos, bias])  # cada fila representa una neurona

# --- Paso 5: Reducción a 3D con PCA
pca = PCA(n_components=3)
espacio_3d = pca.fit_transform(vectores_neuronas)

# --- Paso 6: Visualización
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(espacio_3d[:,0], espacio_3d[:,1], espacio_3d[:,2], c='blue')

ax.set_title("Neuronas en espacio reducido 3D (PCA)")
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
ax.set_zlabel("Componente 3")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# --- Configuración de las neuronas (ejemplo) ---
# Puedes ajustar estos números según tu red real
num_input_neurons = 10    # Cantidad de neuronas de entrada
num_hidden_neurons = 5    # Cantidad de neuronas en la capa oculta
num_output_neurons = 1    # Cantidad de neuronas de salida
embedding_dim = 768       # Dimensión del embedding de cada "palabra" (neurona de entrada)

# --- Generación de embeddings de ejemplo ---
# En un caso real, estas serían las incrustaciones de palabras reales
# para las neuronas de entrada y las características extraídas para las ocultas/salida.
# Aquí se generan aleatoriamente para demostrar la visualización.
input_embeddings = np.random.rand(num_input_neurons, embedding_dim)
hidden_embeddings = np.random.rand(num_hidden_neurons, embedding_dim)
output_embeddings = np.random.rand(num_output_neurons, embedding_dim)

# Concatenar todos los embeddings para procesarlos juntos
all_embeddings = np.concatenate([input_embeddings, hidden_embeddings, output_embeddings])

# --- Asignación de tipos de neuronas y colores ---
neuron_types = ['input'] * num_input_neurons + \
               ['hidden'] * num_hidden_neurons + \
               ['output'] * num_output_neurons

# Diccionario para mapear tipos a colores
color_map = {
    'input': 'green',    # Neuronas de entrada
    'hidden': 'orange',  # Neuronas ocultas
    'output': 'red'      # Neuronas de salida
}
colors = [color_map[neuron_type] for neuron_type in neuron_types]

# --- Reducción de Dimensionalidad con t-SNE ---
# t-SNE es una técnica para reducir datos de alta dimensión a 2D o 3D
# manteniendo la similitud entre los puntos.
# `perplexity` debe ser menor que el número total de muestras.
tsne = TSNE(n_components=2, random_state=0, perplexity=min(5, len(all_embeddings) - 1))
reduced_embeddings = tsne.fit_transform(all_embeddings)

# --- Visualización 2D de las neuronas ---
plt.figure(figsize=(12, 10)) # Tamaño del gráfico

# Dibuja los puntos (neuronas)
plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], c=colors, s=150, alpha=0.8)

# Añade etiquetas para el tipo de neurona
for i, neuron_type in enumerate(neuron_types):
    plt.annotate(neuron_type,
                 xy=(reduced_embeddings[i, 0], reduced_embeddings[i, 1]),
                 xytext=(8, 4), # Desplazamiento del texto
                 textcoords='offset points',
                 ha='left', va='bottom',
                 fontsize=9, color='dimgray')

# Título y etiquetas de los ejes
plt.title('Visualización 2D Espacial de Neuronas (Reducción con t-SNE)', fontsize=16)
plt.xlabel('Dimensión 1 (t-SNE)', fontsize=12)
plt.ylabel('Dimensión 2 (t-SNE)', fontsize=12)

# Crear una leyenda manual
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', label='Entrada (Verde)',
                          markerfacecolor='green', markersize=10),
                   Line2D([0], [0], marker='o', color='w', label='Oculta (Naranja)',
                          markerfacecolor='orange', markersize=10),
                   Line2D([0], [0], marker='o', color='w', label='Salida (Roja)',
                          markerfacecolor='red', markersize=10)]
plt.legend(handles=legend_elements, loc='best', title="Tipo de Neurona")

plt.grid(True, linestyle='--', alpha=0.6) # Añadir una cuadrícula sutil
plt.show()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.manifold import TSNE
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt # Importar matplotlib para visualización

#**1. Obtener Embeddings de BERT y Coordenadas Reducidas:**

def obtener_embeddings_y_coordenadas(texto, modelo_nombre="dccuchile/bert-base-spanish-wwm-cased", n_componentes_tsne=2, random_state=42):
    tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)
    model = AutoModel.from_pretrained(modelo_nombre)

    tokens = tokenizer.tokenize(texto)
    # Asegúrate de que los tokens especiales están escritos correctamente, sin barras invertidas
    tokens_bert = ['[CLS]'] + tokens + ['[SEP]']
    input_ids = tokenizer.convert_tokens_to_ids(tokens_bert)
    input_ids = torch.tensor([input_ids])

    with torch.no_grad():
        outputs = model(input_ids)
        word_embeddings = outputs.last_hidden_state[0]

    # Excluir [CLS] y [SEP]. Asegúrate de que el slicing es correcto.
    # `embeddings_a_visualizar = word_embeddings\[1:-1].numpy()` tenía una barra invertida extra.
    embeddings_a_visualizar = word_embeddings[1:-1].numpy()
    palabras = tokens

    # Ajustar perplexity si el número de muestras es pequeño
    if embeddings_a_visualizar.shape[0] > n_componentes_tsne:
        perplexity_value = min(30, embeddings_a_visualizar.shape[0] - 1)
        # Ensure perplexity is at least 1 if there's at least one sample
        perplexity_value = max(1, perplexity_value)
        tsne = TSNE(n_components=n_componentes_tsne, random_state=random_state, perplexity=perplexity_value)
        coordenadas_reducidas = tsne.fit_transform(embeddings_a_visualizar)
    else:
        # Manejar casos con pocas palabras: devuelve ceros o un array vacío dependiendo de la necesidad.
        # Crear un array de ceros con la forma esperada
        coordenadas_reducidas = np.zeros((embeddings_a_visualizar.shape[0], n_componentes_tsne))
        if embeddings_a_visualizar.shape[0] == 0:
             print("Advertencia: No se extrajeron embeddings de palabras (después de excluir [CLS]/[SEP]). No se puede aplicar t-SNE.")


    return palabras, embeddings_a_visualizar, coordenadas_reducidas

#**2. Definir una Red Neuronal Simple:**

class MiRed(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MiRed, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

#**3. Preparar los Datos y Entrenar (Ejemplo Conceptual):**

# Texto de ejemplo
texto_ejemplo = "El perro ladra fuerte porque ve un gato negro."
palabras, embeddings_bert, coordenadas_tsne = obtener_embeddings_y_coordenadas(texto_ejemplo)

# --- Decidir cómo usar la información para la red ---
combinar_embedding_con_coordenadas = True # Cambiar a False si solo quieres usar el embedding

datos_de_entrada = []
# Asegurarse de iterar sobre el número de embeddings/palabras disponibles
if len(palabras) > 0:
    for i in range(len(palabras)):
        embedding = embeddings_bert[i]
        if combinar_embedding_con_coordenadas:
            # Asegurarse de que tenemos coordenadas para este índice
            if i < coordenadas_tsne.shape[0]:
                coordenada = coordenadas_tsne[i]
                entrada = np.concatenate((embedding, coordenada))
                datos_de_entrada.append(entrada)
            else:
                 print(f"Advertencia: No hay coordenadas t-SNE disponibles para el índice {i}. Saltando.")
        else:
            datos_de_entrada.append(embedding)

# Solo convertir a tensor si hay datos de entrada
if datos_de_entrada:
    datos_de_entrada = torch.tensor(np.array(datos_de_entrada), dtype=torch.float32)

    # --- Crear etiquetas de ejemplo (necesario para el entrenamiento supervisado) ---
    # Esto dependerá de tu tarea. Aquí solo se crean etiquetas aleatorias.
    # Asegurarse de que el número de etiquetas coincide con el número de muestras de entrada.
    etiquetas_ejemplo = torch.randint(0, 2, (datos_de_entrada.shape[0],)).float().unsqueeze(1) # Ejemplo binario

    # --- Definir la red ---
    input_dim_red = datos_de_entrada.shape[1] # Obtener la dimensión de entrada directamente de los datos
    hidden_dim_red = 128
    output_dim_red = 1 # Para clasificación binaria

    modelo = MiRed(input_dim_red, hidden_dim_red, output_dim_red)
    criterio = nn.BCEWithLogitsLoss() # Ejemplo para clasificación binaria
    optimizador = optim.Adam(modelo.parameters(), lr=0.01)

    # --- Bucle de entrenamiento (muy simplificado) ---
    epocas = 10
    for epoca in range(epocas):
        optimizador.zero_grad()
        salida = modelo(datos_de_entrada)
        perdida = criterio(salida, etiquetas_ejemplo)
        perdida.backward()
        optimizador.step()
        print(f'Época {epoca+1}, Pérdida: {perdida.item():.4f}')

    # --- (Opcional) Visualizar los embeddings reducidos ---
    # Solo visualizar si hay coordenadas t-SNE disponibles
    if coordenadas_tsne.shape[0] > 0:
        plt.figure(figsize=(10, 6))
        plt.scatter(coordenadas_tsne[:, 0], coordenadas_tsne[:, 1])
        for i, palabra in enumerate(palabras):
             # Asegurarse de que el índice es válido para las coordenadas
            if i < coordenadas_tsne.shape[0]:
                plt.annotate(palabra, xy=coordenadas_tsne[i], xytext=(5, 2),
                             textcoords='offset points', ha='right', va='bottom')
        plt.title('Visualización t-SNE de Embeddings')
        plt.xlabel('Dimensión 1')
        plt.ylabel('Dimensión 2')
        plt.show()
    else:
        print("No se pudieron visualizar los embeddings t-SNE ya que no se generaron coordenadas.")

else:
    print("No se pudieron preparar datos de entrada para la red neuronal.")

## Capa anti-gradiente (inversión de gradiente)

En el banco: `anti_gradiente`. Ojo: aquí va en el camino principal, así que el extractor aprende a empeorar la misma tarea que el clasificador resuelve; en la industria (DANN, 2015) va en una rama aparte, contra un clasificador de «dominio».

In [ ]:
# train.py
import torch
import torch.nn as nn
import torch.optim as optim

# --- Capa Anti-Gradiente (GRL) ---
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_val=1.0):
        super().__init__()
        self.lambda_val = torch.tensor(lambda_val, requires_grad=False)
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_val)

# --- Modelo Simple con GRL ---
class SimpleAntiGradientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU()
        )
        self.anti_grad_layer = GradientReversalLayer(lambda_val=1.0)
        self.classifier = nn.Sequential(
            nn.Linear(4, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature_extractor(x)
        processed = self.anti_grad_layer(features)
        return self.classifier(processed)

# --- Datos XOR ---
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float32)
y = torch.tensor([[0.],[1.],[1.],[0.]], dtype=torch.float32)

# --- Entrenamiento ---
model = SimpleAntiGradientModel()
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.BCELoss()

for epoch in range(2000):
    optimizer.zero_grad()
    preds = model(X)
    loss = loss_fn(preds, y)
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0:
        print(f"Época {epoch}, Pérdida: {loss.item():.4f}")

# --- Guardar modelo ---
torch.save(model.state_dict(), "/content/antigrad_model.pt")
print("✅ Modelo guardado en /content/antigrad_model.pt")

In [ ]:
# load_model.py
import torch
import torch.nn as nn

# --- Reusar Modelo y GRL ---
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_val):
        ctx.lambda_val = lambda_val
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_val, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_val=1.0):
        super().__init__()
        self.lambda_val = torch.tensor(lambda_val, requires_grad=False)
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_val)

class SimpleAntiGradientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU()
        )
        self.anti_grad_layer = GradientReversalLayer(lambda_val=1.0)
        self.classifier = nn.Sequential(
            nn.Linear(4, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        features = self.feature_extractor(x)
        processed = self.anti_grad_layer(features)
        return self.classifier(processed)

# --- Cargar modelo ---
model = SimpleAntiGradientModel()
model.load_state_dict(torch.load("/content/antigrad_model.pt", weights_only=True))
model.eval()

# --- Evaluación ---
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], dtype=torch.float32)
y = torch.tensor([[0.],[1.],[1.],[0.]], dtype=torch.float32)

with torch.no_grad():
    preds = model(X)
    print("✅ Predicciones:")
    print(f"Predicciones:\n{preds.round()}")
    print(f"Etiquetas reales:\n{y}")

### Variante: anti-gradiente parcial y periódico

Recuperada de tu cuaderno completo (celda 414). En vez de invertir siempre todo el gradiente, invierte el de la mitad de los pesos solo cada 200 épocas. Es un modelo lineal en NumPy sobre XOR (que un modelo lineal no puede resolver), así que sirve para ver la idea, no para medirla.

In [ ]:
import numpy as np

# Datos de ejemplo
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]]) # XOR

# Inicialización de pesos (simulando una capa simple)
W = np.random.rand(2, 1)
b = np.random.rand(1)

learning_rate = 0.1
epochs = 1000

print(f"Pesos iniciales: {W.flatten()}, Sesgo inicial: {b.flatten()}")

for epoch in range(epochs):
    # Forward pass
    output = X @ W + b

    # Cálculo del error (MSE)
    error = y - output

    # Gradiente normal
    dW_normal = X.T @ error * (-1) # Derivada del MSE con respecto a W
    db_normal = np.sum(error) * (-1) # Derivada del MSE con respecto a b

    # Simulando "anti-gradiente" en una porción arbitraria
    # Aquí, por ejemplo, invertimos el gradiente de la primera mitad de los pesos
    # Esto es solo una simulación, no una aplicación real de "anti-gradiente"
    # como se propone en el concepto original.
    dW_anti = dW_normal.copy()
    db_anti = db_normal.copy()

    # Invertir el signo de los gradientes para la primera mitad de los pesos
    dW_anti[0:len(dW_anti)//2] *= -1

    # Podrías decidir cuándo aplicar el anti-gradiente, por ejemplo, cada 100 épocas
    if epoch % 200 == 0:
        W -= learning_rate * dW_anti
        b -= learning_rate * db_anti
        print(f"Época {epoch}: Aplicando Anti-Gradiente. Error: {np.mean(np.abs(error)):.4f}")
    else:
        W -= learning_rate * dW_normal
        b -= learning_rate * db_normal

    if epoch % 100 == 0:
        loss = np.mean(error**2)
        # print(f"Época {epoch}, Pérdida: {loss:.4f}")

print(f"\nPesos finales: {W.flatten()}, Sesgo final: {b.flatten()}")
print(f"Predicciones finales (redondeadas):\n{np.round(X @ W + b)}")
print(f"Valores reales:\n{y}")

## Visualización de grafos de nodos

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def VisualizarGrafoNodos(
    num_nodes: int,
    radius: float,
    min_val_cbar: float,
    max_val_cbar: float,
    cmap_name: str = 'plasma',
    figure_size: tuple = (10, 10),
    title: str = 'Red de Nodos con Barra de Color',
    seed: int = None
):
    """
    Graphus Nodea Visualis: Genera y visualiza un grafo de nodos en 2D,
    coloreados según un valor asociado y con una barra de color personalizada.

    Args:
        num_nodes (int): Número de nodos en el grafo.
        radius (float): Radio máximo para conectar nodos (distancia euclidiana).
        min_val_cbar (float): Valor mínimo para la barra de color y normalización.
        max_val_cbar (float): Valor máximo para la barra de color y normalización.
        cmap_name (str): Nombre del colormap de Matplotlib a usar (ej. 'plasma', 'viridis', 'cividis').
        figure_size (tuple): Tupla (ancho, alto) para el tamaño de la figura.
        title (str): Título del gráfico.
        seed (int, optional): Semilla para la reproducibilidad de la generación de nodos.
                              Si es None, los nodos se generarán aleatoriamente cada vez.
    """
    if seed is not None:
        np.random.seed(seed)

    pos = {}
    node_values = {}
    nodes_list = []

    for i in range(num_nodes):
        node_id = i
        nodes_list.append(node_id)
        pos[node_id] = (np.random.rand(), np.random.rand())
        # Escalar los valores de los nodos para que se ajusten al rango deseado
        # Aquí se asume que los valores se derivan de la coordenada Y,
        # pero en una aplicación real, 'node_values' podría ser una entrada de la función.
        node_values[node_id] = min_val_cbar + (pos[node_id][1] * (max_val_cbar - min_val_cbar))

    G = nx.Graph()
    G.add_nodes_from(nodes_list)

    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            node1 = nodes_list[i]
            node2 = nodes_list[j]
            p1 = np.array(pos[node1])
            p2 = np.array(pos[node2])
            distance = np.linalg.norm(p1 - p2)
            if distance < radius:
                G.add_edge(node1, node2)

    node_colors_values_array = np.array([node_values[node] for node in G.nodes()])

    norm = mcolors.Normalize(vmin=min_val_cbar, vmax=max_val_cbar)
    cmap = plt.cm.get_cmap(cmap_name) # Obtener el colormap por nombre

    node_colors_mapped = cmap(norm(node_colors_values_array))

    plt.figure(figsize=figure_size)
    ax = plt.gca()
    ax.set_facecolor('white')
    ax.set_aspect('equal', adjustable='box')

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gray', alpha=0.1)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors_mapped, node_size=15, ax=ax, linewidths=0.1)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.02, pad=0.03)

    cbar_ticks = np.linspace(min_val_cbar, max_val_cbar, num=6)
    cbar.set_ticks(cbar_ticks)
    cbar.set_ticklabels([f'{t:.2f}' for t in cbar_ticks])
    cbar.set_label('Valor del Nodo', fontsize=12)

    plt.axis('off')
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

## Dibujos de grafos ramificados (networkx)

Vienen del cuaderno 02. Muestran cómo crece un grafo con ramificaciones: nodos principales, ramas y subramas con profundidad.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random

# --- 1. Generación de un grafo más complejo y ramificado ---
G = nx.Graph()
central_node = "Centro"
G.add_node(central_node)

# Función recursiva para añadir ramificaciones con profundidad
def add_branches(graph, parent_node, current_depth, max_depth, branch_factor, node_counter, node_sizes_dict):
    """
    Añade ramificaciones recursivamente a un grafo.
    Modifica node_counter y node_sizes_dict en su lugar.
    """
    if current_depth >= max_depth:
        # Cuando alcanzamos la profundidad máxima, detenemos la recursión
        # y devolvemos el valor actual del contador.
        return node_counter[0] # Devolvemos el valor del contador, no la lista.

    # Calcula el número de ramas para este nodo, con variación aleatoria
    num_branches = random.randint(branch_factor - 1, branch_factor + 1)
    # Asegurarse de que no creamos 0 ramas si no queremos detenernos prematuramente
    num_branches = max(0, num_branches) # Asegura que el número de ramas sea al menos 0

    for _ in range(num_branches):
        node_counter[0] += 1 # Incrementa el contador de nodo
        new_node = f"N{node_counter[0]}" # Crea un nombre de nodo único
        graph.add_node(new_node) # Añade el nuevo nodo al grafo
        graph.add_edge(parent_node, new_node) # Conecta el nuevo nodo con el padre

        # Definir tamaño de nodo basado en profundidad (más profundo = más pequeño)
        if current_depth == 0: # Nodos principales de rama (conectados al centro)
            node_sizes_dict[new_node] = random.randint(800, 1500)
        elif current_depth == max_depth - 1: # Nodos finales ("gotas")
            node_sizes_dict[new_node] = random.randint(50, 250)
        else: # Nodos intermedios
            node_sizes_dict[new_node] = random.randint(200, 600)

        # Llamada recursiva para añadir ramas al nuevo nodo
        # No necesitamos asignar el resultado de vuelta a node_counter[0]
        # porque node_counter es una lista y se modifica en su lugar.
        add_branches(graph, new_node, current_depth + 1, max_depth, branch_factor, node_counter, node_sizes_dict)

    # Devolvemos el valor actual del contador después de añadir todas las ramas para este nivel
    return node_counter[0]


node_counter = [0] # Inicializamos el contador como una lista para pasarlo por referencia
node_sizes_dict = {central_node: 8000} # Tamaño del nodo central
max_depth = 4 # Más niveles de ramificación
branch_factor = 3 # Número promedio de ramas por nodo

# Llamada inicial a la función recursiva.
# La función modificará node_counter[0] y node_sizes_dict en su lugar.
# El valor devuelto por la llamada inicial (el valor final de node_counter) no es estrictamente necesario
# para la lógica siguiente, pero puede ser útil para saber cuántos nodos se crearon.
final_node_count = add_branches(G, central_node, 0, max_depth, branch_factor, node_counter, node_sizes_dict)

# Asignar tamaños de nodo a una lista para nx.draw
# Asegurarse de que cada nodo en G tenga un tamaño definido en node_sizes_dict
node_sizes = [node_sizes_dict.get(node, 100) for node in G.nodes()]

# --- 2. Posicionamiento y Dispersión ---
# Utiliza spring_layout, puedes ajustar k para dispersión y iterations para estabilidad
pos = nx.spring_layout(G, k=0.1, iterations=100, seed=42) # 'k' más bajo para nodos más cercanos, más 'iterations' para mejor convergencia

# --- 3. Colores y Efecto de "Brillo" (Matplotlib) ---
base_color = '#CC0052' # El color principal (rosa/magenta)
glow_color_inner = '#FF6699' # Un tono más claro para el brillo interior
glow_color_outer = '#FFCCE0' # Un tono aún más claro para el brillo exterior
edge_color = '#AA0044' # Un poco más oscuro para las aristas

plt.figure(figsize=(12, 12)) # Aumenta el tamaño de la figura

# Efecto de "brillo": Dibujar varias capas de nodos con diferentes tamaños y transparencias
# Capa más externa (más grande y transparente)
nx.draw_networkx_nodes(G, pos, node_size=[s * 1.5 for s in node_sizes], node_color=glow_color_outer, alpha=0.1, ax=plt.gca())
# Capa intermedia
nx.draw_networkx_nodes(G, pos, node_size=[s * 1.2 for s in node_sizes], node_color=glow_color_inner, alpha=0.3, ax=plt.gca())
# Capa base de nodos
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=base_color, alpha=0.9, ax=plt.gca())

# Dibujar las aristas
nx.draw_networkx_edges(G, pos, edge_color=edge_color, width=0.8, alpha=0.6, ax=plt.gca())

plt.title("Grafo Ramificado Complejo con Efecto de Brillo (Matplotlib)", color='white')
plt.gca().set_facecolor('black') # Fondo oscuro para resaltar el brillo
plt.axis('off')
plt.show()

## Generador de texto recurrente dual (D-ReG)

Viene del cuaderno 02. Dos corrientes que se influyen: «percepción» (atención + red feed-forward) y «conceptualización» (GRU), unidas por unidades de interacción cruzada. Pendiente de medir contra un Transformer y un LSTM en generación de texto (perplejidad).

Ojo: cada atención recibe un solo elemento (secuencia de largo 1), así que el softmax siempre vale 1 y la «atención» no elige nada; habrá que arreglarlo al medirlo.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossInteractionUnit(nn.Module):
    """
    Unidad de Interacción Cruzada (CIU) para permitir que los streams se influyan mutuamente.
    Podría ser una pequeña red de atención o una simple capa lineal y activación.
    """
    def __init__(self, query_dim, key_dim, value_dim, output_dim, num_heads=4):
        super().__init__()
        # Usaremos Multi-Head Attention para permitir una interacción rica
        # Note: embed_dim for MultiheadAttention is the dimension of the query, key, and value after projection
        self.attention = nn.MultiheadAttention(embed_dim=query_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(query_dim)
        self.linear = nn.Linear(query_dim, output_dim) # Para mapear la salida a la dimensión deseada

        # Mapeo lineal para que el "key" y "value" tengan el mismo embed_dim que el "query"
        # Their input dimensions are key_dim and value_dim, output is query_dim
        self.key_proj = nn.Linear(key_dim, query_dim)
        self.value_proj = nn.Linear(value_dim, query_dim) # If key_dim != value_dim

    def forward(self, query_state, influencing_state):
        # query_state: El estado que será influenciado (ej. h_p_t) [batch, query_dim]
        # influencing_state: El estado que influye (ej. h_c_t) [batch, key_dim or value_dim]

        # Expandir las dimensiones para la atención si es necesario (ej. para batch_first=True)
        query = query_state.unsqueeze(1) # [batch, 1, query_dim]
        key = self.key_proj(influencing_state).unsqueeze(1)   # [batch, 1, query_dim]
        value = self.value_proj(influencing_state).unsqueeze(1) # [batch, 1, query_dim]

        # attn_output will be [batch, 1, query_dim]
        attn_output, _ = self.attention(query, key, value)
        attn_output = attn_output.squeeze(1) # [batch, query_dim]

        # Combinar el resultado de la atención con el query original (residual connection)
        # Ensure shapes match for addition: query_state [batch, query_dim] + attn_output [batch, query_dim]
        output = self.norm(query_state + attn_output)

        # Apply final linear layer and activation
        output = F.relu(self.linear(output)) # Output shape: [batch, output_dim]
        return output

class DReGCell(nn.Module):
    """
    La celda central del Generador de Texto Recurrente Dual (D-ReG).
    Se ejecuta una vez por cada token generado.
    """
    def __init__(self, embed_dim, hidden_dim_p, hidden_dim_c, num_heads_p=8, ff_dim_p=2048, num_layers_p=1):
        super().__init__()
        self.embed_dim = embed_dim
        self.hidden_dim_p = hidden_dim_p
        self.hidden_dim_c = hidden_dim_c

        # P-Stream (Percepción): Un bloque de Transformer Decoder (con auto-atención)
        # Aquí simplificamos a un Attention y Feedforward para el ejemplo
        # The attention mechanism operates on the embedding dimension
        self.p_self_attn = nn.MultiheadAttention(embed_dim, num_heads_p, batch_first=True)
        self.p_ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim_p),
            nn.ReLU(),
            nn.Linear(ff_dim_p, embed_dim)
        )
        self.p_norm1 = nn.LayerNorm(embed_dim)
        self.p_norm2 = nn.LayerNorm(embed_dim)

        # C-Stream (Conceptualización): Un GRU o LSTM para mantener el estado conceptual
        # Recibe input del P-Stream y su propio estado anterior
        # Input size should be the dimension of the input it receives (from p_to_c_ciu output)
        # Hidden size should be the dimension of the conceptual state h_c
        # FIX: Set GRU input_size to hidden_dim_c to match p_to_c_ciu output dimension
        self.c_gru = nn.GRUCell(hidden_dim_c, hidden_dim_c) # Input: output de p_to_c_ciu [batch, hidden_dim_c], hidden: h_c_prev [batch, hidden_dim_c]

        # Cross-Interaction Units (CIU)
        # P-Stream influye en C-Stream (para ajustar la conceptualización basada en la percepción)
        # Query: h_c_prev [batch, hidden_dim_c] -> query_dim=hidden_dim_c
        # Influencing: h_p_curr [batch, embed_dim] -> key_dim=embed_dim, value_dim=embed_dim
        # Output needs to be input for GRU -> output_dim=hidden_dim_c
        self.p_to_c_ciu = CrossInteractionUnit(query_dim=hidden_dim_c, key_dim=embed_dim, value_dim=embed_dim, output_dim=hidden_dim_c)
        # C-Stream influye en P-Stream (para guiar la percepción basada en la conceptualización)
        # Query: h_p_prev [batch, embed_dim] -> query_dim=embed_dim
        # Influencing: h_c_prev [batch, hidden_dim_c] -> key_dim=hidden_dim_c, value_dim=hidden_dim_c
        # Output needs to be combined with token embedding -> output_dim=embed_dim
        self.c_to_p_ciu = CrossInteractionUnit(query_dim=embed_dim, key_dim=hidden_dim_c, value_dim=hidden_dim_c, output_dim=embed_dim)

    def forward(self, prev_token_embedding, h_p_prev, h_c_prev):
        # prev_token_embedding: Embedding del token generado en el paso anterior [batch, embed_dim]
        # h_p_prev: Estado oculto anterior del P-Stream [batch, embed_dim]
        # h_c_prev: Estado oculto anterior del C-Stream [batch, hidden_dim_c]

        # 1. P-Stream (Percepción)
        # h_p_prev influenciado por h_c_prev
        # The CIU expects h_p_prev as query_state and h_c_prev as influencing_state
        modulated_h_p_prev = self.c_to_p_ciu(h_p_prev, h_c_prev) # C-Stream guía P-Stream [batch, embed_dim]

        # Input to P-Stream attention: should consider both token embedding and modulated state
        # A simple approach is to combine them, or use them as part of the attention mechanism.
        # Here, we'll use the modulated state as the base and add the token embedding's contribution.
        # A more complex Transformer Decoder would handle this differently.
        # For simplicity in this DReGCell, let's assume the P-Stream self-attention
        # primarily processes the modulated state, guided by the current input token.
        # The line below is a simplification for this cell structure.
        p_input_for_attn = (modulated_h_p_prev + prev_token_embedding).unsqueeze(1) # [batch, 1, embed_dim]


        # Self-attention for the P-Stream.
        # In a decoder real, it would be causal attention over the sequence generated so far.
        # For this cell example, we perform attention on a single timestep input (combined state/token).
        # The output dim of attention is embed_dim, matching the query_dim passed to MultiheadAttention
        p_output_attn, _ = self.p_self_attn(p_input_for_attn, p_input_for_attn, p_input_for_attn) # [batch, 1, embed_dim]
        p_output_attn = p_output_attn.squeeze(1) # [batch, embed_dim]

        # Residual connection and Normalization
        # p_output_attn represents the perception at this step. Combine with previous state influence.
        p_output = self.p_norm1(p_output_attn) # Apply norm after attention

        # Feedforward for P-Stream
        p_output = self.p_norm2(p_output + self.p_ffn(p_output)) # [batch, embed_dim]
        h_p_curr = p_output # This is the new state of the P-Stream [batch, embed_dim]

        # 2. C-Stream (Conceptualization)
        # h_c_prev influenced by h_p_curr
        # The output of the P-Stream influences the C-Stream
        # The CIU expects h_c_prev as query_state and h_p_curr as influencing_state
        # The output dimension of p_to_c_ciu is hidden_dim_c
        c_input_from_p = self.p_to_c_ciu(h_c_prev, h_p_curr) # P-Stream adjusts C-Stream [batch, hidden_dim_c]

        # GRU to update the conceptual state.
        # The input to the GRU is the influence from the P-Stream (c_input_from_p).
        # The hidden state is h_c_prev.
        # Both inputs to GRUCell should have the dimension hidden_dim_c (256)
        h_c_curr = self.c_gru(c_input_from_p, h_c_prev) # [batch, hidden_dim_c]

        return h_p_curr, h_c_curr

class DReG(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim_p, hidden_dim_c, num_heads_enc=8, ff_dim_enc=2048, num_layers_enc=6):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Position embedding size should be sufficient for max sequence length including prompt and generation
        self.position_embedding = nn.Embedding(512 + 50, embed_dim) # Max seq len + max generated len
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.hidden_dim_c = hidden_dim_c # Keep track for initializing h_c_prev

        # Encoder (Transformer estándar) para el prompt inicial
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads_enc, dim_feedforward=ff_dim_enc, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers_enc)

        # The D-ReG cell that will be executed sequentially
        self.dreg_cell = DReGCell(embed_dim, hidden_dim_p, hidden_dim_c)

        # Final projection layer to vocabulary
        # Output is based on the P-Stream state (perception)
        self.output_linear = nn.Linear(hidden_dim_p, vocab_size) # Predict next token from P-Stream state

    def forward(self, input_ids, prompt_mask=None, max_seq_len=50):
        # input_ids: [batch, seq_len_prompt] - the initial prompt
        batch_size = input_ids.size(0)
        seq_len_prompt = input_ids.size(1)

        # 1. Process the prompt with the Encoder
        # Position embeddings should start from 0 for the prompt
        positions = torch.arange(seq_len_prompt, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        input_embeddings = self.token_embedding(input_ids) + self.position_embedding(positions)

        encoder_output = self.encoder(input_embeddings, src_key_padding_mask=prompt_mask)

        # Initialize P-Stream and C-Stream states from encoder output
        # h_p_prev is initialized from the last token of the encoder output [batch, embed_dim]
        h_p_prev = encoder_output[:, -1, :].contiguous()

        # h_c_prev initialized with zeros or a projection from encoder output
        # Its dimension should match hidden_dim_c
        h_c_prev = torch.zeros(batch_size, self.hidden_dim_c, device=input_ids.device) # [batch, hidden_dim_c]

        # The first token embedding for the generation loop is from the last token of the prompt
        next_token_embedding = self.token_embedding(input_ids[:, -1]) # [batch, embed_dim]

        generated_tokens_list = [] # Store generated token IDs step by step

        # 2. Generation Loop
        for i in range(max_seq_len):
            # Execute the D-ReG cell
            # Inputs: next_token_embedding [batch, embed_dim], h_p_prev [batch, embed_dim], h_c_prev [batch, hidden_dim_c]
            h_p_curr, h_c_curr = self.dreg_cell(next_token_embedding, h_p_prev, h_c_prev)

            # Project the output of the P-Stream (h_p_curr) to vocabulary probabilities
            # Output dimension should match hidden_dim_p (or embed_dim depending on model design)
            # FIX: Use h_p_curr which has shape [batch, hidden_dim_p] as input to output_linear
            logits = self.output_linear(h_p_curr) # [batch, vocab_size]
            probabilities = F.softmax(logits, dim=-1)

            # Sample the next token (or take argmax for deterministic inference)
            next_token = torch.argmax(probabilities, dim=-1) # [batch]
            generated_tokens_list.append(next_token.unsqueeze(1)) # [batch, 1]

            # Prepare for the next step
            next_token_embedding = self.token_embedding(next_token) # [batch, embed_dim]
            h_p_prev = h_p_curr # Update P-Stream state for next iteration
            h_c_prev = h_c_curr # Update C-Stream state for next iteration

            # Optional: Stop if EOS token is generated - not implemented here

        # Concatenate the generated tokens
        generated_tokens = torch.cat(generated_tokens_list, dim=1) # [batch, max_seq_len]

        # Combine prompt and generated tokens (optional, depending on desired output format)
        # full_sequence = torch.cat([input_ids, generated_tokens], dim=1) # [batch, seq_len_prompt + max_seq_len]
        return generated_tokens # Return only the generated part


# --- Example Usage and Parameters ---
vocab_size = 10000 # Vocabulary size
embed_dim = 512    # Embedding dimensionality
hidden_dim_p = 512 # Hidden dimension of the P-Stream (Transformer-like)
hidden_dim_c = 256 # Hidden dimension of the C-Stream (GRU-like, conceptual summary)

# Ensure hidden_dim_p matches embed_dim if P-Stream state is embed_dim,
# or adjust output_linear if h_p_curr is hidden_dim_p and it's different from embed_dim.
# Based on DReGCell forward, h_p_curr is embed_dim.
# So output_linear should take embed_dim as input.
# FIX: Change output_linear input dim to embed_dim
class DReG(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim_p, hidden_dim_c, num_heads_enc=8, ff_dim_enc=2048, num_layers_enc=6):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Position embedding size should be sufficient for max sequence length including prompt and generation
        self.position_embedding = nn.Embedding(512 + 50, embed_dim) # Max seq len + max generated len
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.hidden_dim_c = hidden_dim_c # Keep track for initializing h_c_prev

        # Encoder (Transformer estándar) para el prompt inicial
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads_enc, dim_feedforward=ff_dim_enc, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers_enc)

        # The D-ReG cell that will be executed sequentially
        # hidden_dim_p is not explicitly used as state size in DReGCell based on its current forward.
        # h_p_prev and h_p_curr are embed_dim. hidden_dim_p might be intended for ff_dim_p?
        # Let's keep it consistent with the original definition but note the potential discrepancy
        # if hidden_dim_p was meant to be the state size of the P-stream.
        self.dreg_cell = DReGCell(embed_dim, hidden_dim_p, hidden_dim_c) # embed_dim for p_state_size, hidden_dim_p for ff_dim_p

        # Final projection layer to vocabulary
        # Output is based on the P-Stream state (perception), which is h_p_curr
        # h_p_curr has shape [batch, embed_dim] based on DReGCell forward
        # FIX: Output linear takes embed_dim as input
        self.output_linear = nn.Linear(embed_dim, vocab_size) # Predict next token from P-Stream state

    def forward(self, input_ids, prompt_mask=None, max_seq_len=50):
        # input_ids: [batch, seq_len_prompt] - the initial prompt
        batch_size = input_ids.size(0)
        seq_len_prompt = input_ids.size(1)

        # 1. Process the prompt with the Encoder
        # Position embeddings should start from 0 for the prompt
        positions = torch.arange(seq_len_prompt, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        input_embeddings = self.token_embedding(input_ids) + self.position_embedding(positions)

        encoder_output = self.encoder(input_embeddings, src_key_padding_mask=prompt_mask)

        # Initialize P-Stream and C-Stream states from encoder output
        # h_p_prev is initialized from the last token of the encoder output [batch, embed_dim]
        h_p_prev = encoder_output[:, -1, :].contiguous()

        # h_c_prev initialized with zeros or a projection from encoder output
        # Its dimension should match hidden_dim_c
        h_c_prev = torch.zeros(batch_size, self.hidden_dim_c, device=input_ids.device) # [batch, hidden_dim_c]

        # The first token embedding for the generation loop is from the last token of the prompt
        next_token_embedding = self.token_embedding(input_ids[:, -1]) # [batch, embed_dim]

        generated_tokens_list = [] # Store generated token IDs step by step

        # 2. Generation Loop
        for i in range(max_seq_len):
            # Execute the D-ReG cell
            # Inputs: next_token_embedding [batch, embed_dim], h_p_prev [batch, embed_dim], h_c_prev [batch, hidden_dim_c]
            h_p_curr, h_c_curr = self.dreg_cell(next_token_embedding, h_p_prev, h_c_prev)

            # Project the output of the P-Stream (h_p_curr) to vocabulary probabilities
            # h_p_curr has shape [batch, embed_dim]
            logits = self.output_linear(h_p_curr) # [batch, vocab_size]
            probabilities = F.softmax(logits, dim=-1)

            # Sample the next token (or take argmax for deterministic inference)
            next_token = torch.argmax(probabilities, dim=-1) # [batch]
            generated_tokens_list.append(next_token.unsqueeze(1)) # [batch, 1]

            # Prepare for the next step
            next_token_embedding = self.token_embedding(next_token) # [batch, embed_dim]
            h_p_prev = h_p_curr # Update P-Stream state for next iteration
            h_c_prev = h_c_curr # Update C-Stream state for next iteration

            # Optional: Stop if EOS token is generated - not implemented here

        # Concatenate the generated tokens
        generated_tokens = torch.cat(generated_tokens_list, dim=1) # [batch, max_seq_len]

        # Combine prompt and generated tokens (optional, depending on desired output format)
        # full_sequence = torch.cat([input_ids, generated_tokens], dim=1) # [batch, seq_len_prompt + max_seq_len]
        return generated_tokens # Return only the generated part


# --- Example Usage and Parámetros ---
vocab_size = 10000 # Tamaño del vocabulario
embed_dim = 512    # Dimensionalidad de embeddings
hidden_dim_p = 512 # Dimensión oculta del P-Stream (ff_dim_p in DReGCell)
hidden_dim_c = 256 # Dimensión oculta del C-Stream (GRU hidden size)


# Instantiate the model
# The arguments to DReGCell are embed_dim, hidden_dim_p (used as ff_dim_p), hidden_dim_c
model = DReG(vocab_size, embed_dim, hidden_dim_p, hidden_dim_c).to('cpu') # Usar CPU para ejemplo
print(model)

# Example prompt (token IDs)
dummy_prompt_ids = torch.randint(0, vocab_size, (2, 10)) # Batch of 2, prompt of 10 tokens
generated_output = model(dummy_prompt_ids, max_seq_len=20)
print(f"\nForma de la salida generada (IDs de tokens): {generated_output.shape}")

# Parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Número total de parámetros: {total_params}")